In [43]:
#!/usr/bin/env python3
%pip install mlflow lightgbm -q

Note: you may need to restart the kernel to use updated packages.


# Entrenamiento de Modelos de Detección de Clientes con MLFlow

Este notebook entrena múltiples modelos de machine learning con tracking completo en MLFlow.

In [44]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configurar MLFlow
MLFLOW_TRACKING_URI = "http://0.0.0.0:5000"
EXPERIMENT_NAME = "Banco X"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Comprobar si el experimento ya existe y si el artifact store es local y escribible.
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
local_artifact_dir = os.path.join(os.getcwd(), "mlflow_artifacts")
os.makedirs(local_artifact_dir, exist_ok=True)

use_experiment_name = EXPERIMENT_NAME

if exp is not None:
    artifact_location = exp.artifact_location or ""
    # Si el artifact store apunta a un path local, comprobar permisos
    if artifact_location.startswith("file://") or artifact_location.startswith("/"):
        local_path = artifact_location.replace("file://", "")
        try:
            testfile = os.path.join(local_path, ".mlflow_write_test")
            with open(testfile, "w") as f:
                f.write("test")
            os.remove(testfile)
            # si escribible, seguimos usando el experimento existente
        except Exception:
            # No escribible: creamos un nuevo experimento que use la carpeta del workspace
            try:
                use_experiment_name = EXPERIMENT_NAME + "_local"
                mlflow.create_experiment(use_experiment_name, artifact_location=f"file://{local_artifact_dir}")
            except Exception:
                # si falla crear (p. ej. ya existe), nos aseguramos de usar el local_artifact_dir
                pass
else:
    # Experimento no existe: crear apuntando a directorio local para evitar problemas de permisos
    try:
        mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=f"file://{local_artifact_dir}")
    except Exception:
        # si falla por alguna razón, seguir y confiar en el servidor remoto
        pass

# Finalmente seleccionamos el experimento a usar
mlflow.set_experiment(use_experiment_name)

print(f"✓ MLFlow configurado en {MLFLOW_TRACKING_URI}")
print(f"✓ Experimento: {use_experiment_name}")

✓ MLFlow configurado en http://0.0.0.0:5000
✓ Experimento: Client Detection Baseline_local


## Cargar y Dividir Datos

In [46]:
# Cargar datos
data_path = "/workspaces/deteccion_clientes_banco/data/df_resampled.csv"
df = pd.read_csv(data_path, sep=",")

print(f"✓ Datos cargados: {df.shape}")
print(f"  - Filas: {df.shape[0]}")
print(f"  - Columnas: {df.shape[1]}")
print(f"  - Clases: {df['y'].value_counts().to_dict()}")

# Dividir datos
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['y'])
X_train = train.drop("y", axis=1)
Y_train = train["y"]
X_test = test.drop("y", axis=1)
Y_test = test["y"]

print(f"\n✓ Datos divididos:")
print(f"  - Train: {X_train.shape}")
print(f"  - Test: {X_test.shape}")

✓ Datos cargados: (46548, 28)
  - Filas: 46548
  - Columnas: 28
  - Clases: {0: 36548, 1: 10000}

✓ Datos divididos:
  - Train: (37238, 27)
  - Test: (9310, 27)


## Definir Funciones de Entrenamiento y Evaluación

In [47]:
def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    """Calcula métricas de clasificación"""
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    
    if y_pred_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_pred_proba)
    
    return metrics


def train_model(model_name, model_config, X_train, Y_train, X_test, Y_test, log_model=True):
    """Entrena un modelo y trackea en MLFlow. Si log_model=False no intentará subir el modelo."""
    
    print(f"\n{'='*60}")
    print(f"Entrenando: {model_name}")
    print(f"{'='*60}")
    
    with mlflow.start_run(run_name=model_name):
        try:
            # Crear pipeline
            if model_name == "Logistic Regression":
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(**model_config))
                ])
            elif model_name == "Random Forest":
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", RandomForestClassifier(**model_config))
                ])
            elif model_name == "XGBoost":
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", XGBClassifier(**model_config))
                ])
            elif model_name == "LightGBM":
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", lgb.LGBMClassifier(**model_config))
                ])
            else:
                raise ValueError(f"Modelo desconocido: {model_name}")
            
            # Entrenar
            pipe.fit(X_train, Y_train)
            
            # Predicciones
            y_pred = pipe.predict(X_test)
            y_pred_train = pipe.predict(X_train)
            y_pred_proba = pipe.predict_proba(X_test)[:, 1]
            y_pred_proba_train = pipe.predict_proba(X_train)[:, 1]
            
            # Calcular métricas
            test_metrics = calculate_metrics(Y_test, y_pred, y_pred_proba)
            train_metrics = calculate_metrics(Y_train, y_pred_train, y_pred_proba_train)
            
            # Log de parámetros
            mlflow.log_params(model_config)
            
            # Log de métricas
            for metric_name, metric_value in test_metrics.items():
                mlflow.log_metric(f"test_{metric_name}", metric_value)
            
            for metric_name, metric_value in train_metrics.items():
                mlflow.log_metric(f"train_{metric_name}", metric_value)
            
            # Log del modelo (safe)
            if log_model:
                try:
                    # Guardamos el pipeline con mlflow.sklearn (funciona con Pipeline)
                    input_example = X_test.head(3)
                    mlflow.sklearn.log_model(pipe, artifact_path="model", input_example=input_example)
                except Exception as e:
                    # Si falla el log de modelo, lo registramos como artifact local (pickle)
                    print(f"! Warning: no se pudo loggear el modelo con mlflow.sklearn: {e}")
                    tmp_dir = os.path.join(os.getcwd(), "tmp_model_artifacts")
                    os.makedirs(tmp_dir, exist_ok=True)
                    model_file = os.path.join(tmp_dir, f"{model_name.replace(' ', '_')}_pipeline.pkl")
                    try:
                        import pickle
                        with open(model_file, "wb") as f:
                            pickle.dump(pipe, f)
                        mlflow.log_artifact(model_file, artifact_path="model_pickle")
                    except Exception as e2:
                        print(f"✗ Error guardando artifact local del modelo: {e2}")
            
            # Imprimir resultados
            print(f"\n✓ Modelo entrenado exitosamente")
            print(f"\nMétricas de Test:")
            for metric_name, metric_value in test_metrics.items():
                print(f"  {metric_name}: {metric_value:.4f}")
            
            print(f"\nMétricas de Train:")
            for metric_name, metric_value in train_metrics.items():
                print(f"  {metric_name}: {metric_value:.4f}")
            
            # Detectar overfitting
            overfitting_gap = train_metrics['accuracy'] - test_metrics['accuracy']
            print(f"\nBrecha Accuracy (Train - Test): {overfitting_gap:.4f}")
            
            return pipe, test_metrics, train_metrics
            
        except Exception as e:
            print(f"✗ Error entrenando {model_name}: {str(e)}")
            try:
                mlflow.log_param("error", str(e))
            except Exception:
                pass
            return None, None, None

## Entrenar Modelos y Trackear en MLFlow


### Setear Hiperparámetros

In [ ]:
# Definir configuraciones de modelos
models_config = {
    "XGBoost": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 6,
        "eval_metric": "logloss",
        "random_state": 42,
        "verbosity": 0
    },
    "Random Forest": {
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "random_state": 42,
        "n_jobs": -1
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 6,
        "num_leaves": 31,
        "random_state": 42,
        "verbose": -1
    },
    "Logistic Regression": {
        "max_iter": 1000,
        "random_state": 42,
        "solver": "lbfgs"
    }
}




Entrenando: XGBoost


2025/12/04 16:50:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



✓ Modelo entrenado exitosamente

Métricas de Test:
  accuracy: 0.9323
  precision: 0.8571
  recall: 0.8220
  f1: 0.8392
  roc_auc: 0.9789

Métricas de Train:
  accuracy: 0.9406
  precision: 0.8781
  recall: 0.8400
  f1: 0.8586
  roc_auc: 0.9838

Brecha Accuracy (Train - Test): 0.0082
🏃 View run XGBoost at: http://0.0.0.0:5000/#/experiments/2/runs/2f4ea96b9a4b42df9e0a4ce3795645b8
🧪 View experiment at: http://0.0.0.0:5000/#/experiments/2

✓ Entrenamiento completado para 1/1 modelos


### Entrenar Modelos

None, "XGBoost", "LightGBM", "Logistic Regression", "Random Forest"

In [ ]:
# CONTROL: seleccionar un único modelo para ejecutar o None para ejecutar todos
SELECT_MODEL = "XGBoost"  # poner e.g. "XGBoost" para ejecutar solo XGBoost
LOG_MODEL = True     # si False, no se intentará subir el modelo a MLFlow (evita errores de artifact store)


results = {}
to_run = [SELECT_MODEL] if SELECT_MODEL is not None else list(models_config.keys())

for model_name in to_run:
    config = models_config.get(model_name)
    if config is None:
        print(f"! Modelo '{model_name}' no encontrado en models_config. Saltando.")
        continue

    pipe, test_metrics, train_metrics = train_model(
        model_name, config, X_train, Y_train, X_test, Y_test, log_model=LOG_MODEL
    )
    
    if pipe is not None:
        results[model_name] = {
            "pipe": pipe,
            "test_metrics": test_metrics,
            "train_metrics": train_metrics
        }

print(f"\n{'='*60}")
print(f"✓ Entrenamiento completado para {len(results)}/{len(to_run)} modelos")
print(f"{'='*60}")

## MLFlow Tracking

Todas las corridas han sido registradas en MLFlow. Puedes acceder a:
- **URL**: http://0.0.0.0:5000
- **Experimento**: Banco X
- **Modelos registrados**: 4 corridas independientes (una por cada modelo)

Cada corrida contiene:
- Hiperparámetros del modelo
- Métricas de test (accuracy, precision, recall, f1, roc_auc)
- Métricas de train
- Modelo serializado para reproducibilidad